In [ ]:
# EXT-KN01 (extension, spec v1.2 "Extension"): Docling HybridChunker conversion, 4 extension chunkers, bge-m3 vectors for
# the new chunk texts, scorer_v1 on the new settings only. Tuning documents only (dataset EXT-KD01); no --final anywhere.
# Needs: GPU T4, Internet on, dataset EXT-KD01 attached. No secrets (no LLM chunker in this run).
COMMIT = "e26e916"
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git checkout -q {COMMIT} && git log --oneline -1 && git status --porcelain | head

In [ ]:
# Same pins as P2-KN03 / P3-KN04, plus docling-core 2.106.0 (the version in the docling_hybrid chunker's version string)
!pip install -q docling==2.143.0 "docling-core[chunking]==2.106.0" jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 google-genai==2.34.0
!pip freeze > /kaggle/working/ext-kn01_pip_freeze.txt
!grep -i -E "^(docling|docling-core|torch|transformers|sentence-transformers|semchunk)==" /kaggle/working/ext-kn01_pip_freeze.txt
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Copy EXT-KD01 into the clone at its repo paths. Every file must be listed in BUNDLE.json with the same SHA-256 and
# belong to a tuning document; anything else stops the notebook before a run.
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

REPO, INPUT, TEMP = Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/temp/ext-kd01")
sys.path.insert(0, str(REPO))
from src.common.manifest import code_version
from src.common.split import tuning_ids

record_path = next(INPUT.rglob("BUNDLE.json"))
bundle = record_path.parent
if not (bundle / "data").is_dir():  # uploaded with --dir-mode zip and not unpacked by Kaggle
    for archive in bundle.glob("*.zip"):
        zipfile.ZipFile(archive).extractall(TEMP / archive.stem)  # the CLI zips a folder without its own name
    bundle = TEMP
record, tuning = json.loads(record_path.read_text()), tuning_ids()
assert set(record["documents"]) == tuning, "EXT-KD01 documents differ from the tuning split of this commit"
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
for entry in record["files"]:
    source, target = bundle / entry["path"], REPO / entry["path"]
    assert Path(entry["path"]).stem in tuning and sha(source) == entry["sha256"], entry["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
for path in [*(bundle / "data" / "gold").glob("*.jsonl"), *(bundle / "data" / "embeddings" / "bge-m3").iterdir()]:
    target = REPO / path.relative_to(bundle)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(path, target)
gold_docs = {json.loads(line)["doc_id"] for path in (REPO / "data" / "gold").glob("*.jsonl") for line in path.open()}
assert gold_docs <= tuning, sorted(gold_docs - tuning)
for folder in ["raw", "extracted/docling", "extracted/pdfplumber", "gold", "embeddings/bge-m3"]:
    print(folder, ":", sum(path.is_file() for path in (REPO / "data" / folder).rglob("*")), "files")
print("tuning documents:", len(tuning), "| code:", code_version())

In [ ]:
# Step 1: re-convert the tuning PDFs with the frozen converter (GPU), compare with the frozen text, run HybridChunker.
!cd /kaggle/working/repo && python -m scripts.ext_docling_hybrid_convert --machine kaggle --run-source EXT-KN01
import pandas as pd
table_path = max((REPO / "data" / "results" / "ext_docling_hybrid").glob("text_comparison_*.csv"))
shutil.copyfile(table_path, Path("/kaggle/working") / table_path.name)
comparison = pd.read_csv(table_path)
print(table_path.name, "| identical texts:", int(comparison["text_identical"].sum()), "of", len(comparison))
comparison

In [ ]:
# Step 2: the four extension chunkers on all tuning documents (names as in the laptop smoke tests).
import subprocess

RUNS = [("paragraph_group", "", "docling"), ("dynamic_token", "", "docling pdfplumber"),
        ("section_table", "--heading-prefix", "docling pdfplumber"), ("docling_hybrid", "", "docling")]
failed = []
for chunker, flags, parsers in RUNS:
    command = f"python -m scripts.p2_chunk --chunker {chunker} {flags} --parsers {parsers} --machine kaggle --run-source EXT-KN01"
    result = subprocess.run(command, shell=True, cwd=REPO, capture_output=True, text=True)
    lines = (result.stdout + result.stderr).strip().splitlines()
    print(f"=== {chunker} {flags} {parsers} | exit {result.returncode}\n" + "\n".join(lines[:1] + lines[-3:]) + "\n")
    if result.returncode:
        failed.append(chunker)
print("failed:", failed or "none")
!ls /kaggle/working/repo/data/chunks/*

In [ ]:
# Step 3: bge-m3 vectors (GPU, float32, pinned revision) for the new chunk texts; the tuning questions come cached.
!cd /kaggle/working/repo && python -m scripts.p3_embed --name EXT-KN01-new --machine kaggle --run-source EXT-KN01

In [ ]:
# Step 4: scorer_v1 on the new settings only, as a new run P3-EXT-KN01-...
SETTINGS = "paragraph_group_noprefix dynamic_token_noprefix section_table_prefix docling_hybrid_noprefix"
!cd /kaggle/working/repo && python -m scripts.p3_score --settings {SETTINGS} --machine kaggle --run-source EXT-KN01

In [ ]:
# Step 5: one zip of every output and run record, at repo paths (unzip at the repo root on the laptop).
OUTPUTS = ["data/extracted/docling_hybrid", "data/extracted/docling_doc", "data/results/ext_docling_hybrid", "data/chunks",
           "data/embeddings/bge-m3/EXT-KN01-new.npy", "data/embeddings/bge-m3/EXT-KN01-new.keys.json"]
files = [path for name in OUTPUTS for path in ([REPO / name] if (REPO / name).is_file() else sorted((REPO / name).rglob("*")))
         if path.is_file()]
files += sorted((REPO / "data" / "results" / "p3").glob("P3-EXT-KN01-*")) + sorted((REPO / "runs").glob("P?-EXT-KN01-*.json"))
with zipfile.ZipFile("/kaggle/working/ext-kn01_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(REPO))
    archive.write("/kaggle/working/ext-kn01_pip_freeze.txt", "data/external/kaggle_out/EXT-KN01/ext-kn01_pip_freeze.txt")
print(len(files), "files zipped | run records:", sorted(path.name for path in files if path.parent.name == "runs"))
!ls -la /kaggle/working